# Traffic-law expert system — evaluation (Kaggle)
Settings: **GPU T4 x2**, **Internet on**. Private repo: Add-ons → Secrets → `GH_TOKEN` (attach).
Run 1: `RUN_PLAN = ["A", "B", "C"]` → Save & Run All. Run 2: `RUN_PLAN = ["D", "E"]`. Download `results.zip` from Output.

## Config

In [ ]:
REF = "eval-v1"
LIMIT = None                      # e.g. 5 for a smoke test
CASES = "data/eval/test_cases.jsonl"
REPO_URL = "https://github.com/LoylP/traffic-law-expert-system.git"

MAIN_MODEL = "qwen3.5:4b-q4_K_M"
MODELS = {
    "A": "qwen3.5:4b-q4_K_M",
    "B": "gemma4:e4b",
    "C": "llama3.2:3b",
    "D": "bonsai-2:4b",
    "E": "qwen3.5:9b-q4_K_M",
}
SMALL_TIER = ["A", "B", "C", "D"]

RUN_PLAN = ["A", "B", "C"]        # run 1: A, B, C | run 2: D, E
ABLATE_SMALL = True
TIME_BUDGET_H = 11.0

COMPARE_CONFIGS = ["full", "llm_only"]
ABLATION_CONFIGS = ["full", "no_derive", "no_circumstance", "no_grounder", "llm_only"]
NO_LLM_CONFIGS = ["oracle_facts", "regex_extractor"]
LIMIT_ARG = "" if LIMIT is None else f"--limit {LIMIT}"
OUT = "/kaggle/working"

## GPU

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Repo

In [ ]:
import os
if not os.path.isdir("/tmp/repo"):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        token = None
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    !git clone -q --branch {REF} {url} /tmp/repo
    !git -C /tmp/repo remote set-url origin {REPO_URL}
%cd /tmp/repo
!git log -1 --oneline --decorate
!pip install -q -r requirements.txt

## Ollama

In [ ]:
import subprocess, time, requests
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

def ollama_up():
    try:
        return requests.get("http://localhost:11434/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_up():
    log = open("/tmp/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_CONTEXT_LENGTH": "8192"},
                     stdout=log, stderr=log)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
assert ollama_up(), "Ollama not reachable on :11434, see /tmp/ollama.log"
!ollama --version

## Env & output paths

In [ ]:
from pathlib import Path
Path(".env").write_text("\n".join([
    "LLM_PROVIDER=ollama",
    "LLM_BASE_URL=http://localhost:11434/v1",
    "LLM_SEED=42",
    "LLM_DISABLE_THINKING=true",
    "LLM_CACHE_DIR=data/eval/cache",
]) + "\n")

for local, target in {"data/eval/results": Path(OUT) / "results", "data/eval/cache": Path(OUT) / "cache"}.items():
    local = Path(local)
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        !cp -rn {local}/. {target}/ && rm -rf {local}
    local.symlink_to(target, target_is_directory=True)
    print(f"{local} -> {target}")

## Helpers

In [ ]:
import sys, re
from tqdm import tqdm
ip = get_ipython()

NOISE = ("HF_TOKEN", "Redirects are currently", "Loading weights", "huggingface_hub")
START = re.compile(r"\] (\d+)/(\d+) đã có, còn (\d+) case")
STEP = re.compile(r"\] (\d+)/(\d+) (\S+) \(([\d.]+)s\)")

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h{sec % 3600 // 60:02d}m" if sec >= 3600 else f"{sec // 60}m{sec % 60:02d}s"

def eval_cmd(model, config, *extra):
    cmd = [sys.executable, "-u", "-m", "src.eval.run_eval", "--cases", CASES, "--config", config,
           *extra, *LIMIT_ARG.split()]
    return cmd + (["--model", model] if model else [])

def run_eval_live(model, config):
    proc = subprocess.Popen(eval_cmd(model, config, "--resume"), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    bar, tail, t0 = None, [], time.time()
    for line in proc.stdout:
        tail = (tail + [line])[-40:]
        if m := START.search(line):
            bar = tqdm(total=int(m[2]), initial=int(m[1]), desc=f"{config:>16}", unit="case",
                       mininterval=30, bar_format="{desc} {bar} {n}/{total} [{elapsed}<{remaining}] {postfix}")
        elif (m := STEP.search(line)) and bar:
            bar.update(1)
            bar.set_postfix_str(f"{m[3]} {m[4]}s")
        elif not any(n in line for n in NOISE):
            print(line, end="")
    code = proc.wait()
    if bar:
        bar.close()
    if code:
        print(f"\n[FAIL] {config} (exit {code}), last log lines:\n" + "".join(tail))
    return code, time.time() - t0

def is_complete(model, config):
    return subprocess.run(eval_cmd(model, config, "--check-complete"), capture_output=True).returncode == 0

def run_model(model, configs):
    todo = [c for c in configs if not is_complete(model, c)]
    if not todo:
        print(f"{model}: complete, skipped")
        return
    t0 = time.time()
    print(f"> {model}: {todo} (start {time.strftime('%H:%M')} UTC)")
    ip.system(f"ollama pull {model} 2>&1 | tail -n 1")
    report = []
    for i, config in enumerate(todo, 1):
        print(f"\n[{i}/{len(todo)}] {model} | {config} | elapsed {fmt(time.time() - t0)}")
        code, sec = run_eval_live(model, config)
        report.append((config, "OK" if code == 0 else "FAIL", fmt(sec)))
    ip.system(f"ollama rm {model}")
    print(f"\n# {model} done in {fmt(time.time() - t0)} (end {time.strftime('%H:%M')} UTC)")
    for config, state, dur in report:
        print(f"  {config:>16}  {state:4}  {dur}")

def status():
    ablate = {MODELS[c] for c in SMALL_TIER} if ABLATE_SMALL else {MAIN_MODEL}
    print(f"{'':3}{'model':22}" + "".join(f"{c:>17}" for c in ABLATION_CONFIGS))
    for cell, model in MODELS.items():
        cfgs = ABLATION_CONFIGS if model in ablate else COMPARE_CONFIGS
        marks = ["x" if is_complete(model, c) else ("." if c in cfgs else "") for c in ABLATION_CONFIGS]
        print(f"{cell:3}{model:22}" + "".join(f"{m:>17}" for m in marks))
    print(f"{'':3}{'no-LLM':22}" + "".join(f"{c}: {'x' if is_complete(None, c) else '.'}   " for c in NO_LLM_CONFIGS))
    print("x = done, . = pending")

status()

## Run

In [ ]:
t_start = time.time()
print(f"Plan: {RUN_PLAN} | start {time.strftime('%H:%M')} UTC")
if "A" in RUN_PLAN:
    for config in NO_LLM_CONFIGS:
        run_eval_live(None, config)
for cell in RUN_PLAN:
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print(f"Time budget reached, skipping {cell} and later cells")
        break
    configs = ABLATION_CONFIGS if ABLATE_SMALL and cell in SMALL_TIER else COMPARE_CONFIGS
    print(f"\n===== {cell}: {MODELS[cell]} =====")
    try:
        run_model(MODELS[cell], configs)
    except Exception as error:
        print(f"[FAIL] {cell}: {error}")
print(f"\nTotal: {fmt(time.time() - t_start)}")
status()

## Aggregate

In [ ]:
!python -m src.eval.aggregate --main-model {MAIN_MODEL}
!cp -r data/eval/report {OUT}/report
!cd {OUT} && zip -qr results.zip results report && ls -lh {OUT}